In [2]:
import time
import numpy as np


class Transform3D:

    def __init__(self, R: np.ndarray, p: np.ndarray):
        # Construcción de la matriz de transformación homogénea 4x4.
        # R: Matriz de rotación (3x3) p: Vector de traslación
        
        self.R = np.array(R, dtype=np.float64)
        self.p = np.array(p, dtype=np.float64).reshape(3, 1)

        # Construcción de la matriz homogénea T (4x4)
        self.T = np.eye(4, dtype=np.float64)
        self.T[:3, :3] = self.R
        self.T[:3, 3:] = self.p

    def inv_analytic(self) -> np.ndarray:
        # Calculo de la inversa analítica: [R^T, -R^T * p; 0, 1]
        R_T = self.R.T
        p_inv = -R_T @ self.p

        T_inv = np.eye(4, dtype=np.float64)
        T_inv[:3, :3] = R_T
        T_inv[:3, 3:] = p_inv
        return T_inv

    def inv_generic(self) -> np.ndarray:
        # Calculo de la inversa numérica
        return np.linalg.inv(self.T)


In [3]:
# 1 y 2. Definición de la matriz de prueba (SO(3) válida)

# Rotación de 45° en Z como ejemplo representativo
theta = np.pi / 4
R_test = np.array(
    [
        [np.cos(theta), -np.sin(theta), 0.0],
        [np.sin(theta), np.cos(theta), 0.0],
        [0.0, 0.0, 1.0],
    ]
)
p_test = np.array([1.5, -2.0, 3.0])
transform = Transform3D(R_test, p_test)

In [4]:
# 3. Validación de Ortogonalidad / Error
T = transform.T
T_inv = transform.inv_analytic()
I4 = np.eye(4)

# Cálculo de la matriz de error E = T * T^-1 - I
E = (T @ T_inv) - I4
norm_F = np.linalg.norm(E, ord="fro")

print("=" * 50)
print("VALIDACIÓN NUMÉRICA")
print("=" * 50)
print(f"Norma de Frobenius ||E||_F: {norm_F:.4e}")
print(f"¿Cumple ||E||_F < 1e-14?:   {norm_F < 1e-14}")
print()

VALIDACIÓN NUMÉRICA
Norma de Frobenius ||E||_F: 7.0217e-16
¿Cumple ||E||_F < 1e-14?:   True



In [5]:
# 4. Benchmark Temporal (100,000 iteraciones)
N_ITER = 100_000

# Benchmark Método Genérico
t0 = time.perf_counter()
for _ in range(N_ITER):
    _ = transform.inv_generic()
t_generic = time.perf_counter() - t0

# Benchmark Método Analítico
t0 = time.perf_counter()
for _ in range(N_ITER):
    _ = transform.inv_analytic()
t_analytic = time.perf_counter() - t0

# Reducción porcentual
reduccion_pct = ((t_generic - t_analytic) / t_generic) * 100

print("=" * 50)
print(f"BENCHMARK TEMPORAL ({N_ITER:,} ejecuciones)")
print("=" * 50)
print(f"{'Método':<25} | {'Tiempo Total (s)':<18}")
print("-" * 50)
print(f"{'Genérico (np.linalg.inv)':<25} | {t_generic:<18.5f}")
print(f"{'Analítico ([R^T, -R^T*p])':<25} | {t_analytic:<18.5f}")
print("-" * 50)
print(f"Reducción porcentual de tiempo: {reduccion_pct:.2f}%\n")

BENCHMARK TEMPORAL (100,000 ejecuciones)
Método                    | Tiempo Total (s)  
--------------------------------------------------
Genérico (np.linalg.inv)  | 1.02557           
Analítico ([R^T, -R^T*p]) | 0.70611           
--------------------------------------------------
Reducción porcentual de tiempo: 31.15%

